# Chapter 04 — Recreate Results Figures
Run after kernel restart. No model training required. Set ROOT to your PhishingDetection project folder. Missing optional files are reported, not invented.

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt

# Works when the notebook is run from PhishingDetection/ OR PhishingDetection/notebooks/
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if p.name.lower() == 'phishingdetection'), HERE)
OUT = ROOT / 'results' / 'report_figures'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 250})

def locate(name):
    # Look in typical experiment output folders first, then anywhere inside project
    candidates = [ROOT / name, ROOT / 'results' / name,
                  ROOT / 'results' / 'distilbert_full_dataset' / name,
                  ROOT / 'results' / 'xai_200_full_distilbert' / name,
                  ROOT / 'data' / 'processed' / name,
                  ROOT / 'dataset' / 'processed' / name]
    for p in candidates:
        if p.is_file():
            print(f'FOUND {name}: {p.relative_to(ROOT)}')
            return p
    matches = list(ROOT.rglob(name))
    if matches:
        print(f'FOUND {name}: {matches[0].relative_to(ROOT)}')
        return matches[0]
    print(f'MISSING {name} under {ROOT}')
    return None

def save(name):
    plt.tight_layout()
    plt.savefig(OUT / name, bbox_inches='tight')
    plt.show()
    plt.close()

print('Notebook working directory:', HERE)
print('Project root:', ROOT)
print('Saving figures to:', OUT)


In [ ]:
# Figure 4.1 — Original class distribution (source: original dataset counts)
counts = {'Benign':428103,'Defacement':96457,'Phishing':94111,'Malware':32520}
fig, ax = plt.subplots(figsize=(7,4)); ax.bar(counts.keys(),counts.values())
ax.set_ylabel('Number of URLs'); ax.set_title('Original URL Class Distribution')
for i,v in enumerate(counts.values()): ax.text(i,v+4500,f'{v:,}',ha='center',fontsize=9)
save('Figure_4_1_Class_Distribution.png')

In [ ]:
# Figure 4.2 — URL lengths, only when saved test CSV is present
p = locate('test.csv') or locate('test_data.csv') or locate('test_data_clean.csv')
if p:
    d=pd.read_csv(p); col='clean_url' if 'clean_url' in d else 'url'
    if col in d and 'label_binary' in d:
        groups=[d.loc[d.label_binary==i,col].dropna().astype(str).str.len() for i in (0,1)]
        fig,ax=plt.subplots(figsize=(6,4));ax.boxplot(groups,tick_labels=['Benign','Malicious'],showfliers=False)
        ax.set_ylabel('URL length (characters)');ax.set_title('URL Length by Binary Class')
        save('Figure_4_2_URL_Length.png')
    else: print('Expected URL and label_binary columns not found')
else: print('Figure 4.2 skipped: export your held-out test data to test.csv first.')

In [ ]:
# Figure 4.3 — Final DistilBERT confusion matrix
p=locate('test_metrics.json')
if p:
    m=json.loads(p.read_text()); cm=np.array(m['confusion_matrix'])
    fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(cm,cmap='Blues')
    ax.set_xticks([0,1],['Benign','Malicious']);ax.set_yticks([0,1],['Benign','Malicious'])
    ax.set_xlabel('Predicted label');ax.set_ylabel('True label');ax.set_title('Full DistilBERT Confusion Matrix')
    for (i,j),v in np.ndenumerate(cm):ax.text(j,i,f'{v:,}',ha='center',va='center',color='white' if v>cm.max()/2 else 'black')
    fig.colorbar(im,ax=ax);save('Figure_4_3_Confusion_Matrix.png')

In [ ]:
# Figure 4.5 — Model metrics, use final full-dataset JSON separately
files={'Random Forest':'random_forest_metrics.json','XGBoost':'xgboost_metrics.json','Initial DistilBERT':'distilbert_metrics.json','Full DistilBERT':'test_metrics.json'}
rows=[]
for label,name in files.items():
    p=locate(name)
    if p:
        v=json.loads(p.read_text());rows.append({'Model':label,**{k:100*v[k] for k in ['accuracy','precision','recall','roc_auc']},'f1':100*v.get('f1',v.get('f1_score',0))})
if rows:
    df=pd.DataFrame(rows).set_index('Model');fig,ax=plt.subplots(figsize=(10,5))
    df[['accuracy','precision','recall','f1']].plot.bar(ax=ax)
    ax.set_ylim(0,105);ax.set_ylabel('Score (%)');ax.set_title('Held-out Test Classification Performance');ax.legend(title='Metric');ax.tick_params(axis='x',rotation=15)
    save('Figure_4_5_Model_Comparison.png')

In [ ]:
# Figures 4.4 and 4.6 — read actual XAI summary, never estimate values
p=locate('summary_by_method.csv')
if p:
    x=pd.read_csv(p); print('Available XAI columns:',x.columns.tolist());display(x.head())
    method_col=next((c for c in ['method','Method','xai_method'] if c in x),None)
    if method_col:
        x=x.set_index(method_col)
        faith=[c for c in ['occlusion_spearman_mean','deletion_probability_drop_mean'] if c in x]
        if faith:
            fig,ax=plt.subplots(figsize=(7,4));x[faith].plot.bar(ax=ax);ax.set_title('XAI Faithfulness');ax.set_ylabel('Mean metric');ax.tick_params(axis='x',rotation=0);save('Figure_4_4_XAI_Faithfulness.png')
        if 'explain_seconds_mean' in x:
            fig,ax=plt.subplots(figsize=(6,4));x['explain_seconds_mean'].plot.bar(ax=ax);ax.set_title('XAI Explanation Time');ax.set_ylabel('Seconds per explanation');ax.tick_params(axis='x',rotation=0);save('Figure_4_6_XAI_Runtime.png')
    else: print('Method column not identified; inspect printed CSV columns.')
else: print('XAI figures skipped: copy summary_by_method.csv into project results folder.')

In [ ]:
# Optional HTTPS-prefix flip-rate figure
p=locate('https_prefix_robustness.csv')
if p:
    d=pd.read_csv(p);rates=d.groupby('actual_label')['prediction_flipped'].mean().reindex([0,1])*100
    fig,ax=plt.subplots(figsize=(6,4));ax.bar(['Benign','Malicious'],rates)
    ax.set_ylim(0,100);ax.set_ylabel('Prediction flip rate (%)');ax.set_title('HTTPS-Prefix Sensitivity')
    for i,v in enumerate(rates):ax.text(i,v+2,f'{v:.1f}%',ha='center')
    save('Optional_HTTPS_Prefix_Flip_Rate.png')
print('All available figures saved to',OUT)